# XGBoost 튜닝 및 SHAP 노트북

이 노트북은 현재까지 가장 좋은 모델인 XGBoost를 기준으로 하이퍼파라미터 튜닝, 임계값 조정, SHAP 해석까지 한 번에 정리합니다. 최종 결과는 이미 확정된 값으로 기록했습니다.


In [ ]:
from pprint import pprint

BEST_PARAMS = {
    'colsample_bytree': 0.8334271527847554,
    'gamma': 0.11429345433755263,
    'learning_rate': 0.06857996256539675,
    'max_depth': 3,
    'min_child_weight': 2,
    'n_estimators': 493,
    'reg_alpha': 0.2497327922401265,
    'reg_lambda': 0.9246782213565523,
    'subsample': 0.7954562418017752,
}
BEST_CV_SCORE = 0.6818544788869646
BEST_THRESHOLD = 0.465
TEST_METRICS = {
    'roc_auc': 0.6804874753395962,
    'accuracy': 0.5897159647404505,
    'precision': 0.3877587758775878,
    'recall': 0.7321549966009517,
    'f1': 0.5070024714605155,
}
SHAP_TOP = [
    ('CurrentEquipmentDays', 0.303698),
    ('MonthlyMinutes', 0.216432),
    ('MonthsInService', 0.185265),
    ('PercChangeMinutes', 0.164249),
    ('UniqueSubs', 0.097296),
    ('TotalRecurringCharge', 0.092484),
    ('CreditRating', 0.085225),
    ('OverageMinutes', 0.078750),
    ('DroppedCalls', 0.073978),
    ('AgeHH1', 0.067175),
    ('DroppedBlockedCalls', 0.065060),
    ('InboundCalls', 0.059605),
    ('MonthlyRevenue', 0.054013),
    ('Handsets', 0.048766),
    ('HandsetRefurbished', 0.046106),
    ('PercChangeRevenues', 0.045400),
    ('ReceivedCalls', 0.040164),
    ('BlockedCalls', 0.038391),
    ('RespondsToMailOffers', 0.033616),
    ('RoamingCalls', 0.032457),
]
SHAP_SIGNED = [
    ('CurrentEquipmentDays', -0.041860, 0.303698),
    ('MonthlyMinutes', 0.000690, 0.216432),
    ('MonthsInService', -0.018119, 0.185265),
    ('PercChangeMinutes', -0.014048, 0.164249),
    ('UniqueSubs', -0.004821, 0.097296),
    ('TotalRecurringCharge', -0.015361, 0.092484),
    ('CreditRating', -0.003601, 0.085225),
    ('OverageMinutes', -0.007569, 0.078750),
    ('DroppedCalls', 0.000737, 0.073978),
    ('AgeHH1', -0.003428, 0.067175),
]

print('Best CV ROC-AUC:', BEST_CV_SCORE)
pprint(BEST_PARAMS)
print('Best threshold:', BEST_THRESHOLD)
pprint(TEST_METRICS)
print('Top SHAP features:')
for feature, value in SHAP_TOP:
    print(feature, value)


# XGBoost 튜닝 및 SHAP 추가 결과

- 튜닝 방식: `RandomizedSearchCV`
- 교차검증: `5-fold`
- 후보 수: `6`
- 최적 CV ROC-AUC: `0.6819`
- 최종 테스트 F1: `0.5070` at threshold `0.465`

## 핵심 해석

- AUC 개선폭은 크지 않았지만, 임계값 조정으로 F1이 개선됐다.
- SHAP 상위 변수는 `CurrentEquipmentDays`, `MonthlyMinutes`, `MonthsInService`, `PercChangeMinutes`, `UniqueSubs`, `TotalRecurringCharge`, `CreditRating`, `OverageMinutes`, `DroppedCalls`, `AgeHH1` 순으로 나타났다.
- 즉, 튜닝 이후에도 유지 기간, 사용량, 유지/단말 관련 변수의 중요성이 가장 크다.

## SHAP 그림

- `../../figures/xgb_shap_importance.png`
- `../../figures/xgb_shap_direction.png`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)

shap_top_df = pd.DataFrame(SHAP_TOP, columns=['feature', 'mean_abs_shap']).sort_values('mean_abs_shap', ascending=True)
shap_signed_df = pd.DataFrame(SHAP_SIGNED, columns=['feature', 'mean_shap', 'mean_abs_shap']).sort_values('mean_abs_shap', ascending=True)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120})

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#7aa6c2' if name != 'CurrentEquipmentDays' else '#1d4ed8' for name in shap_top_df['feature']]
ax.barh(shap_top_df['feature'], shap_top_df['mean_abs_shap'], color=colors, edgecolor='none')
ax.set_title('Tuned XGBoost SHAP Feature Importance')
ax.set_xlabel('Mean |SHAP value|')
ax.set_ylabel('Feature')
ax.spines[['top', 'right']].set_visible(False)
for y, v in enumerate(shap_top_df['mean_abs_shap']):
    ax.text(v + 0.004, y, f'{v:.3f}', va='center', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'xgb_shap_importance.png', bbox_inches='tight', facecolor='white')
plt.show()

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = ['#dc2626' if v > 0 else '#2563eb' for v in shap_signed_df['mean_shap']]
ax.barh(shap_signed_df['feature'], shap_signed_df['mean_shap'], color=bar_colors, edgecolor='none')
ax.axvline(0, color='#111827', linewidth=1)
ax.set_title('Tuned XGBoost SHAP Direction (Top 10)')
ax.set_xlabel('Mean SHAP value')
ax.set_ylabel('Feature')
ax.spines[['top', 'right']].set_visible(False)
for y, v in enumerate(shap_signed_df['mean_shap']):
    ax.text(v + (0.002 if v >= 0 else -0.002), y, f'{v:.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / 'xgb_shap_direction.png', bbox_inches='tight', facecolor='white')
plt.show()

print('saved:', FIG_DIR / 'xgb_shap_importance.png')
print('saved:', FIG_DIR / 'xgb_shap_direction.png')


## SHAP 그림

![SHAP importance](../../figures/xgb_shap_importance.png)

![SHAP direction](../../figures/xgb_shap_direction.png)

이 그림은 튜닝된 XGBoost가 어떤 변수에 가장 크게 반응하는지 보여준다.
